# NewsSeek GPU Encoding Pipeline (CLIP ViT-B/32 vs SigLIP ViT-B/16-224)

Notebook này đóng gói toàn bộ quy trình **In-Memory Streaming Encoding** cho tập **NewsSeek (698 video tin tức đa ngôn ngữ)** trên Google Colab GPU và xuất trực tiếp kho vector FAISS chuẩn cho server FastAPI local.

### Trước khi chạy:
1. Vào **Runtime → Change runtime type → Hardware accelerator → GPU** (ưu tiên **A100 / L4**; **T4** miễn phí vẫn chạy tốt nhờ cơ chế **Checkpoint & Auto-Resume** lưu thẳng vào Google Drive).
2. Chuẩn bị **Kaggle API Token** của bạn (lấy tại https://www.kaggle.com/settings/api). Khuyên dùng **Colab Secrets** (biểu tượng chìa khóa bên trái → thêm secret tên `KAGGLE_API_TOKEN`).
3. Nên chạy thử với `MAX_VIDEOS = 20` (~3–5 phút) trước để kiểm tra mọi thứ hoạt động trơn tru, rồi đổi thành `MAX_VIDEOS = 0` để chạy full 698 video.

## Bước 1: Kiểm tra GPU & Mount Google Drive (lưu kết quả + checkpoint an toàn)

In [ ]:
!nvidia-smi

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/xlav_newsseek_output')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output & Checkpoints will be saved safely to: {DRIVE_OUTPUT_DIR}')

## Bước 2: Cài đặt đầy đủ thư viện & Nạp Kaggle API Token an toàn

In [ ]:
!pip install -q faiss-cpu transformers sentencepiece protobuf opencv-python-headless pillow kaggle tqdm pandas

import os
from pathlib import Path
from getpass import getpass

token = None
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    pass

if not token:
    token = getpass('Nhập Kaggle API Token của bạn (ẩn khi gõ): ').strip()

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
token_file = kaggle_dir / 'access_token'
token_file.write_text(token, encoding='utf-8')
os.chmod(token_file, 0o600)
os.environ['KAGGLE_API_TOKEN'] = token
print('Kaggle credentials configured successfully!')

## Bước 3: Clone mã nguồn dự án từ GitHub
*(Nếu bạn đã push bản sửa `scripts/colab_run_newsseek_a100.py` mới lên repo, cell này sẽ lấy bản mới nhất. Nếu chưa push, bạn cũng có thể upload trực tiếp file `colab_run_newsseek_a100.py` vào `/content/repo/scripts/`)*

In [ ]:
REPO_URL = 'https://github.com/labfitius/clip-siglip-semantic-retrieval.git'
REPO_DIR = Path('/content/clip-siglip-semantic-retrieval')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd /content/clip-siglip-semantic-retrieval

## Bước 4: Chạy Pipeline Streaming In-Memory + Tự động lưu Checkpoint lên Drive

- `MAX_VIDEOS = 20`: Chạy thử nhanh 20 video (~3–5 phút) để kiểm chứng end-to-end.
- `MAX_VIDEOS = 0`: Chạy toàn bộ **698 video** (~60 phút trên A100, ~3 tiếng trên T4).
- Nếu Colab bị ngắt kết nối giữa chừng, chỉ cần **chạy lại từ Bước 1 → Bước 4**, script sẽ tự đọc checkpoint trong `DRIVE_OUTPUT_DIR/checkpoints/` và chạy tiếp từ video đang dở!

In [ ]:
MAX_VIDEOS = 20        # Đổi thành 0 để encode toàn bộ 698 video
BATCH_SIZE = 128       # A100 có thể tăng lên 256; T4/L4 giữ 128
CHECKPOINT_EVERY = 50  # Lưu checkpoint lên Google Drive sau mỗi 50 video

!python3 scripts/colab_run_newsseek_a100.py \
    --raw-dir /content/newsseek_raw \
    --output-dir /content/drive/MyDrive/xlav_newsseek_output \
    --batch-size {BATCH_SIZE} \
    --max-videos {MAX_VIDEOS} \
    --checkpoint-every {CHECKPOINT_EVERY}

## Bước 5: Xác minh sản phẩm trên Google Drive trước khi tải về máy Local

In [ ]:
import json
import faiss
from pathlib import Path

out = Path('/content/drive/MyDrive/xlav_newsseek_output')
zip_file = out / 'newsseek_vector_stores.zip'
bm_file = out / 'newsseek_benchmark_results.json'

assert zip_file.exists(), 'Không tìm thấy newsseek_vector_stores.zip!'
print(f'ZIP ready: {zip_file} ({zip_file.stat().st_size / (1024**2):.2f} MB)')

clip_idx = faiss.read_index(str(out / 'storage/newsseek/clip/index.faiss'))
siglip_idx = faiss.read_index(str(out / 'storage/newsseek/siglip/index.faiss'))
print(f'CLIP FAISS vectors  : {clip_idx.ntotal:,} (dim={clip_idx.d})')
print(f'SigLIP FAISS vectors: {siglip_idx.ntotal:,} (dim={siglip_idx.d})')

if bm_file.exists():
    print('\n--- Benchmark Summary ---')
    print(json.dumps(json.loads(bm_file.read_text(encoding='utf-8')), indent=2))

## Bước 6: Hướng dẫn nạp về máy Local (macOS)

Sau khi Bước 5 báo thành công, tải file `MyDrive/xlav_newsseek_output/newsseek_vector_stores.zip` về máy Mac rồi chạy trong Terminal:

```bash
cd /Users/phungquochuy/Desktop/xlav_midterm
unzip ~/Downloads/newsseek_vector_stores.zip -d .
```

Sau đó khởi động lại FastAPI server (`python3 -m uvicorn server:app --host 127.0.0.1 --port 8000`) và mở http://127.0.0.1:8000 để truy vấn trực tiếp trên tab **NewsSeek**!